Minor Project


In [58]:
import pandas as pd
import numpy as np
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("imdbdataset.csv")

In [30]:
df.drop_duplicates(inplace=True)

Text-Preprocessing

In [31]:
#converting to lowercase

df["review"] = df["review"].str.lower()

In [32]:
#Deleting URLs

import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text

df["review"] = df["review"].apply(remove_urls)

In [33]:
#Removing Punctuations

def remove_punc(text):
    text = re.sub(r"[^\w\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punc)

In [34]:
#Removing HTML tags

def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [45]:
#Removing Stopwords

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [43]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)


In [ ]:
#stemming 

def stemming(text):
    ps = PorterStemmer()
    stemwords = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemwords.append(stemmed_tokens)
    
    return " ".join(stemwords)

df["review"] = df["review"].apply(stemming)

In [ ]:
#encoding

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])
df

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1
...,...,...
49995,thought th move dd rght good job t wsnt s cret...,1
49996,bd plot bd dlogu bd ctng dotc drectng nnoyng p...,0
49997,ctholc tught n prochl elentri school nun tught...,0
49998,im gog dgree previou comnt side mlt e secd rte...,0


In [ ]:
y = df["sentiment"]

In [65]:
#vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4056560 stored elements and shape (49582, 5000)>

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [66]:
X_train = X_train.toarray()
X_test = X_test.toarray()


In [69]:
train_set = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float())
test_set = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test.values).float())


In [71]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

Build RNN

In [83]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x)
        out = self.fc(out[:,-1:,:])
        return out

In [84]:
model = RNN(X_train.shape[1])
optimiser = optim.Adam(model.parameters())
criterion = nn.BCELoss()


Train Model

In [85]:
epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0.0
    for xb, yb in train_loader:
        train_loss=0.0
        optimiser.zero_grad()
        xb = xb.unsqueeze(1) #adding 1 dimension
        output = model(xb)
        output = torch.sigmoid(output.squeeze()) #removing 1 dimension
        loss = criterion(output, yb)
        loss.backward()
        optimiser.step()
        train_loss +=loss
    epoch_train_loss = train_loss/len(train_loader)
    print(f"{epoch+1}/{epochs} ==> train loss = {epoch_train_loss}")


1/10 ==> train loss = 0.0003502176550682634
2/10 ==> train loss = 0.0005880852113477886
3/10 ==> train loss = 0.0004439262265805155
4/10 ==> train loss = 0.0006147260428406298
5/10 ==> train loss = 0.0003346903540659696
6/10 ==> train loss = 0.00039183645276352763
7/10 ==> train loss = 0.00039342103991657495
8/10 ==> train loss = 0.0004159638483542949
9/10 ==> train loss = 0.0005323342047631741
10/10 ==> train loss = 0.00048215370043180883


In [87]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb,yb in test_loader:
        optimiser.zero_grad()
        xb = xb.unsqueeze(1) #adding 1 dimension
        output = model(xb)
        output = torch.sigmoid(output.squeeze()) #removing 1 dimension
        loss = criterion(output, yb)
        predicted = (output>0.5).float()
        total += yb.size(0)
        correct += (predicted==yb).sum().item()
    
    print(f"accuracy = {correct/total*100} %")

accuracy = 85.74165574266411 %
